# SWE015 · Use Case 4 — Real-Time Chat Moderation (Turkish)
Runs the whole model ladder end to end. **Runtime → Change runtime type → T4 GPU** before starting.

Total time ≈ 25–35 min (BERTurk fine-tuning ≈ 10–15 min on a T4).

In [ ]:
# 1. Get the code
!git clone https://github.com/Yamfy/swe015-chat-moderation.git
%cd swe015-chat-moderation
!pip install -q -r requirements.txt

In [ ]:
# 2. Data: OffensEval-TR 2020 (official train -> 90/10 train/val, official test = held-out test)
!python -m src.data

In [ ]:
# 3. Area B: lexicon + edit-distance rules, tf-idf + logistic regression
!python train_classical.py

In [ ]:
# 4. Area A: own word2vec (skip-gram, negative sampling) + feedforward net, SGD + backprop
!python train_nn.py

In [ ]:
# 5. Area C: fine-tune BERTurk (GPU)
!python train_bert.py

In [ ]:
# 6. Area D: dynamic INT8 quantization
!python quantize_bert.py

In [ ]:
# 7. Evaluate ALL models on the same test set; latency measured on CPU (>=100 single-message runs)
!python evaluate.py

In [ ]:
# 8. Upload weights to the Hugging Face Hub (repo stays free of large files)
#    Add your HF write token in Colab: key icon (Secrets) -> name HF_TOKEN
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
HF_REPO = 'YOUR_HF_USERNAME/swe015-chat-moderation'   # <- change
!python upload_weights.py {HF_REPO}

In [ ]:
# 9. Download the results folder (needed for the slides + to commit results/ to GitHub)
!zip -r results.zip results
from google.colab import files
files.download('results.zip')

In [ ]:
# 10. (Optional) try the app here with a public link
!python app.py --share